<a href="https://colab.research.google.com/github/amit-eyal/apono-assignment/blob/main/sql_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import pandas as pd
import sqlite3
import plotly.express as px

df = pd.read_csv('/content/JIT_Homework_Data.csv')
conn = sqlite3.connect('sql_analysis.db')

# checking for duplicates in the customerID
dups_query = 'SELECT CustomerID, COUNT(*) from customers GROUP BY CustomerID HAVING COUNT(*) > 1'
check_id_dups_df = pd.read_sql_query(dups_query, conn)
print(len(check_id_dups_df))

# checking for null values and info
df.to_sql('customers', conn, if_exists= 'replace', index=False)
print(df.isnull().sum())
df.info()
df.describe(include='all')
df.head()

0
CustomerID                          0
AccountType                         0
RenewalProbability                  0
SupportTickets                      0
Policy EnforcementUsage             0
Policy EnforcementSatisfaction      0
JIT Token GenerationUsage           0
JIT Token GenerationSatisfaction    0
Access LogsUsage                    0
Access LogsSatisfaction             0
Compliance ReportingUsage           0
Compliance ReportingSatisfaction    0
Admin DashboardUsage                0
Admin DashboardSatisfaction         0
Access Delays                       0
Compliance Questions                0
Integration Issues                  0
Other                               0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 18 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   CustomerID                        100 non-null    object 
 1   Acc

,CustomerID,AccountType,RenewalProbability,SupportTickets,Policy EnforcementUsage,Policy EnforcementSatisfaction,JIT Token GenerationUsage,JIT Token GenerationSatisfaction,Access LogsUsage,Access LogsSatisfaction,Compliance ReportingUsage,Compliance ReportingSatisfaction,Admin DashboardUsage,Admin DashboardSatisfaction,Access Delays,Compliance Questions,Integration Issues,Other
0,CUST1,Mid-Market,51.57,7,0.34,5,0.91,5,0.35,2,0.00,3,0.27,5,2,0,4,2
1,CUST2,Enterprise,81.82,3,0.38,2,0.11,3,0.40,1,0.33,1,0.59,3,4,4,4,4
2,CUST3,Enterprise,65.72,0,0.09,5,0.49,1,0.10,2,0.40,2,0.36,5,1,4,4,4
3,CUST4,Mid-Market,75.43,7,0.58,3,0.01,1,0.74,3,0.54,3,0.09,2,3,2,2,3
4,CUST5,Small Business,95.38,3,0.04,5,0.47,5,0.18,5,0.92,1,0.92,3,1,1,3,4


In [57]:
# 1) Feature Renewals

# filtering for Enterprise customers only
query = 'SELECT * FROM customers WHERE(customers.AccountType= "Enterprise")'
enterprise_df = pd.read_sql_query(query, conn)
list_columns = [c for c in enterprise_df.columns if 'Satisfaction' in c or 'Usage' in c]+ ['RenewalProbability']
print(list_columns)
# filtering based on the relevant fields only
enterprise_df = enterprise_df[list_columns]
features_set = set([f.replace('Usage', '').replace('Satisfaction', '') for f in list_columns if 'Usage' in f or 'Satisfaction' in f])
print(features_set)
# calculating Feature Scores and Droping old columns
for f in features_set:
  usage_col = f + 'Usage'
  sat_col = f + 'Satisfaction'
  enterprise_df[f'{f}_score'] = enterprise_df[usage_col] * enterprise_df[sat_col]
  enterprise_df = enterprise_df.drop(columns=[usage_col, sat_col])

# calculating Correlation and sorting the data
renewal_corr = enterprise_df.corr(numeric_only=True)
renewal_corr = renewal_corr['RenewalProbability'].sort_values(ascending=False).drop('RenewalProbability')
print(renewal_corr)

# creating a bar graph for data visualization
correlation_fig = px.bar(renewal_corr, x= renewal_corr.index.str.replace('_score', ''), y= renewal_corr, labels={'x': 'Feature Name', 'y': 'Correlation'}, title= 'Correlation of Features with Renewal Probability', text=renewal_corr.values)
correlation_fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')

correlation_fig.show()

['Policy EnforcementUsage', 'Policy EnforcementSatisfaction', 'JIT Token GenerationUsage', 'JIT Token GenerationSatisfaction', 'Access LogsUsage', 'Access LogsSatisfaction', 'Compliance ReportingUsage', 'Compliance ReportingSatisfaction', 'Admin DashboardUsage', 'Admin DashboardSatisfaction', 'RenewalProbability']
{'Admin Dashboard', 'JIT Token Generation', 'Compliance Reporting', 'Policy Enforcement', 'Access Logs'}
Admin Dashboard_score         0.346131
Policy Enforcement_score      0.188828
Compliance Reporting_score    0.123431
Access Logs_score             0.007500
JIT Token Generation_score   -0.158313
Name: RenewalProbability, dtype: float64


In [42]:
# 2) Adoption Improvement

# filtering for Mid-Market customers only
query = 'SELECT * FROM customers WHERE(customers.AccountType= "Mid-Market")'
mid_market_df = pd.read_sql_query(query, conn)
usage_columns_list = [c for c in mid_market_df.columns if 'Usage' in c]
# filtering based on the relevant fields only
usage_featurs_df = mid_market_df[usage_columns_list]
# calculating the average usage for each feature
usage_featurs_df = usage_featurs_df.mean().sort_values(ascending=True)
usage_featurs_df.index = [f"Avg {c.replace('Usage', '').strip('_')}" for c in usage_featurs_df.index]
print(usage_featurs_df)

# creating a horizontal bar graph for data visualization
usage_fig = px.bar(usage_featurs_df, orientation='h', labels= {'value': 'Average Usage', 'index': 'Feature Name'}, title= 'Average Usage of Features', text=usage_featurs_df.values)
usage_fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')

usage_fig.show()


Avg Admin Dashboard         0.435000
Avg Policy Enforcement      0.453056
Avg Compliance Reporting    0.502500
Avg Access Logs             0.596944
Avg JIT Token Generation    0.636944
dtype: float64


In [56]:
# 3) Support Tickets

# suming up the total tickets in each category
tickets_query = 'SELECT SUM("Access Delays") AS sum_Access_Delays, SUM("Compliance Questions") AS sum_Compliance_Questions, SUM("Integration Issues") AS sum_Integration_Issues, SUM(Other) As sum_Other\
 FROM customers'
ticket_breakdowns_df = pd.read_sql_query(tickets_query, conn).iloc[0]
# sorting the categories from highest to lowest to idenify what category needs improving
ticket_breakdowns_df = ticket_breakdowns_df.sort_values(ascending=False)
print(ticket_breakdowns_df)

# creating a horizontal bar graph for data visualization
ticket_breakdowns_fig = px.bar(ticket_breakdowns_df, orientation='h', labels= {'value': 'Number of Tickets', 'index': 'Ticket Category'}, title= 'Ticket Breakdowns By Category', text=ticket_breakdowns_df.values)
ticket_breakdowns_fig.update_traces(texttemplate='%{text:f}', textposition='outside')

ticket_breakdowns_fig.show()


sum_Integration_Issues      215
sum_Access_Delays           211
sum_Other                   194
sum_Compliance_Questions    182
Name: 0, dtype: int64
